# Proyecto 4: Análisis Real - Impacto del Tipo de Cambio en Precios

## Fase 1: Extracción de Datos mediante la API del BCRP
En vez de descargar Excels manualmente (como hace la mayoría), vamos a construir un pipeline que consuma automáticamente la API pública del Banco Central de Reserva del Perú (BCRP).

Extraeremos dos variables clave en frecuencia mensual:
1. **Tipo de Cambio (Venta)**
2. **Índice de Precios al Consumidor (IPC)**

In [1]:
import urllib.request
import json
import ssl
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configuraciones visuales
sns.set_theme(style="darkgrid")
plt.rcParams['figure.figsize'] = (12, 6)

### 1.1 Configuración de la Conexión a la API
El Banco Central suele tener problemas con sus certificados SSL. Usaremos un contexto inseguro para evitar errores de conexión (`ssl.CERT_NONE`), tal como observamos en tu archivo de prueba anterior.

In [2]:
ctx = ssl.create_default_context()
ctx.check_hostname = False
ctx.verify_mode = ssl.CERT_NONE

headers = {'User-Agent': 'Mozilla/5.0'}

### 1.2 Extracción de Datos
El endpoint de BCRP requiere los códigos de las series. Por defecto pediremos datos de los últimos 10 años (2014 - 2024).

In [3]:
def obtener_datos_bcrp(series_codes, start_period, end_period):
    """
    Descarga datos de la API del BCRP para múltiples series estructuradas.
    """
    url = f"https://estadisticas.bcrp.gob.pe/estadisticas/series/api/{series_codes}/json/{start_period}/{end_period}"
    req = urllib.request.Request(url, headers=headers)
    
    print(f"[INFO] Consultando API: {url}")
    
    with urllib.request.urlopen(req, context=ctx) as response:
        data = json.loads(response.read().decode('utf-8'))
    return data

# Usaremos los códigos oficiales del BCRP para Tipo de Cambio e IPC:
# PN01206PM = Tipo de Cambio (S/ por US$) - Venta Interbancaria
# PN01271PM = Índice de precios Lima Metropolitana (var% mensual) - IPC
codigos = "PN01206PM-PN01271PM"

# Obtenemos datos desde Enero 2014 hasta Diciembre 2024
datos_brutos = obtener_datos_bcrp(codigos, "2014-01", "2024-12")
print("\nExtracción Exitosa. Mostrando estructura parcial:")
print(json.dumps(datos_brutos['config'], indent=2))

[INFO] Consultando API: https://estadisticas.bcrp.gob.pe/estadisticas/series/api/PN01206PM-PN01271PM/json/2014-01/2024-12

Extracción Exitosa. Mostrando estructura parcial:
{
  "title": "Tipo de cambio - promedio del periodo (S/ por US$)",
  "series": [
    {
      "name": "Tipo de cambio - promedio del periodo (S/ por US$) - Interbancario - Venta",
      "dec": "3"
    },
    {
      "name": "\u00cdndice de precios Lima Metropolitana (var% mensual) - IPC",
      "dec": "2"
    }
  ]
}


### 1.3 Transformación de JSON a DataFrame (Pandas)
La respuesta del BCRP viene en un formato JSON anidado en `periods`. Vamos a limpiarlo y estructurarlo en un DataFrame tabular y asignar el índice de tiempo correctamente.

In [4]:
filas = []

for registro in datos_brutos['periods']:
    # El formato del BCRP es "Ene.2014"
    fecha_str = registro['name']
    # Los valores numéricos de las series vienen en una lista 'values'
    valores = registro['values']
    
    filas.append({
        'fecha': fecha_str,
        'tipo_cambio': float(valores[0]) if valores[0] != 'n.d.' else None,
        'ipc': float(valores[1]) if valores[1] != 'n.d.' else None
    })

df = pd.DataFrame(filas)

# Mostrar los primeros registros
df.head()

,fecha,tipo_cambio,ipc
0,Ene.2014,2.810195,0.316847
1,Feb.2014,2.813355,0.600839
2,Mar.2014,2.807252,0.518559
3,Abr.2014,2.795065,0.393222
4,May.2014,2.787948,0.225030


### 1.4 Parseo de Fechas Peruanas
El formato `Ene.2014` no es estándar para Pandas. Haremos un reemplazo de diccionarios para transformar los meses al inglés temporalmente y poder convertirlos a `datetime`.

In [5]:
meses_es_en = {
    'Ene': 'Jan', 'Feb': 'Feb', 'Mar': 'Mar', 'Abr': 'Apr',
    'May': 'May', 'Jun': 'Jun', 'Jul': 'Jul', 'Ago': 'Aug',
    'Sep': 'Sep', 'Oct': 'Oct', 'Nov': 'Nov', 'Dic': 'Dec'
}

def convertir_fecha(fecha_str):
    mes, anio = fecha_str.split('.')
    mes_en = meses_es_en.get(mes, mes)
    return f"{anio}-{mes_en}-01"

df['fecha'] = df['fecha'].apply(convertir_fecha)
df['fecha'] = pd.to_datetime(df['fecha'])
df.set_index('fecha', inplace=True)

print(df.info())
df.head()

<class 'pandas.DataFrame'>
DatetimeIndex: 132 entries, 2014-01-01 to 2024-12-01
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   tipo_cambio  132 non-null    float64
 1   ipc          132 non-null    float64
dtypes: float64(2)
memory usage: 3.1 KB
None


,tipo_cambio,ipc
fecha,,
2014-01-01,2.810195,0.316847
2014-02-01,2.813355,0.600839
2014-03-01,2.807252,0.518559
2014-04-01,2.795065,0.393222
2014-05-01,2.787948,0.225030


¡Perfecto! Ya tenemos un DataFrame profesional de series de tiempo. Vamos a guardar esto en disco para la base de nuestro modelado analítico.

In [6]:
import os
os.makedirs("../data", exist_ok=True)
df.to_csv("../data/bcrp_tc_ipc_mensual.csv")
print("Datos guardados exitosamente en: data/bcrp_tc_ipc_mensual.csv")

Datos guardados exitosamente en: data/bcrp_tc_ipc_mensual.csv
